In [ ]:
# !pip install pingouin -q  # uncomment if not installed

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix,
    cohen_kappa_score
)
import pingouin as pg
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)

LABEL_CSV  = '../Result/Label_base human.csv'
VISUAL_CSV = '../Result/Visual_base human.csv'

RESULT_DIR = '../Result/Analyze_Results'
os.makedirs(RESULT_DIR, exist_ok=True)

CLASSES = ['Left rotate', 'Normal', 'Right rotate']
ENCODE  = {'Left rotate': 1, 'Normal': 2, 'Right rotate': 3}

In [ ]:
# ─── Helper functions ────────────────────────────────────────────────────────

def normalize(label):
    """Strip whitespace and canonicalize to one of 3 class names."""
    if pd.isna(label): return np.nan
    s = str(label).strip()
    if s.lower().startswith('left'):  return 'Left rotate'
    if s.lower().startswith('right'): return 'Right rotate'
    if s.lower() == 'normal':         return 'Normal'
    return np.nan


def load_and_clean(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    rater_cols_orig = [c for c in df.columns if c not in ('Image', 'GT')]
    for col in ['GT'] + rater_cols_orig:
        df[col] = df[col].apply(normalize)
    rename_map = {col: f'R{i+1:02d}' for i, col in enumerate(rater_cols_orig)}
    df = df.rename(columns=rename_map)
    name_map = {rename_map[c]: c for c in rater_cols_orig}   # R-code -> original name
    return df, [rename_map[c] for c in rater_cols_orig], name_map


def specificity_macro(y_true, y_pred):
    """Macro-averaged specificity (TN / (TN+FP)) across all 3 classes."""
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    specs = []
    for i in range(len(CLASSES)):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - TP - FP - FN
        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return float(np.mean(specs))


def rater_metrics(y_true, y_pred):
    return {
        'Accuracy':    accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score   (y_true, y_pred, average='macro', zero_division=0),
        'Specificity': specificity_macro(y_true, y_pred),
        'Precision':   precision_score (y_true, y_pred, average='macro', zero_division=0),
        'F1':          f1_score        (y_true, y_pred, average='macro', zero_division=0),
    }


def compute_icc(df, rater_cols):
    """Build long-format table and run pingouin ICC (human raters only)."""
    rows = []
    for _, row in df.iterrows():
        for rater in rater_cols:
            val = ENCODE.get(row[rater], np.nan)
            if not np.isnan(val):
                rows.append({'targets': row['Image'], 'raters': rater, 'ratings': float(val)})
    return pg.intraclass_corr(
        data=pd.DataFrame(rows), targets='targets', raters='raters', ratings='ratings'
    )


def build_metrics_table(df, rater_cols):
    records = []
    for rater in rater_cols:
        mask = df[rater].notna() & df['GT'].notna()
        m = rater_metrics(df.loc[mask, 'GT'].tolist(), df.loc[mask, rater].tolist())
        m['Rater'] = rater
        records.append(m)
    return pd.DataFrame(records).set_index('Rater')[
        ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1']
    ]


def display_with_summary(tbl, title):
    print(f'\n{"+"*72}\n  {title}\n{"+"*72}')
    mean = tbl.mean()
    std  = tbl.std(ddof=1)
    summary = {c: f'{mean[c]:.3f} \u00b1 {std[c]:.3f}' for c in tbl.columns}
    disp = pd.concat([
        tbl.round(3).astype(str),
        pd.DataFrame([summary], index=['Mean \u00b1 SD'])
    ])
    display(disp)
    return mean, std


def display_icc(icc_df, title):
    print(f'\n{"+"*72}\n  {title}\n{"+"*72}')
    cols = ['Type', 'Description', 'ICC', 'F', 'df1', 'df2', 'pval', 'CI95%']
    display(icc_df[cols].round(3))

In [ ]:
from itertools import combinations


def compute_weighted_kappa(df, rater_cols):
    """Mean pairwise Cohen's quadratic-weighted kappa among human raters.

    Ordinal labels are encoded Left=1, Normal=2, Right=3. For every pair of
    raters we use only the images both of them scored, so raters who labelled
    different subsets still contribute. Quadratic weights make Left<->Right
    disagreements count 4x a one-step (Left<->Normal) miss -- the same ordinal
    penalty the ICC uses, so this number should track the ICC closely.

    Returns (summary_dict, pairwise_matrix_df).
    """
    enc = pd.DataFrame({r: df[r].map(ENCODE) for r in rater_cols})
    mat = pd.DataFrame(np.nan, index=rater_cols, columns=rater_cols)
    kappas = []
    for a, b in combinations(rater_cols, 2):
        mask = enc[a].notna() & enc[b].notna()
        if mask.sum() < 2:
            continue
        ya = enc.loc[mask, a].astype(int)
        yb = enc.loc[mask, b].astype(int)
        k = cohen_kappa_score(ya, yb, labels=[1, 2, 3], weights='quadratic')
        if np.isnan(k):
            continue
        mat.loc[a, b] = mat.loc[b, a] = k
        kappas.append(k)
    kappas = np.array(kappas)
    summary = {
        'mean':    kappas.mean(),
        'sd':      kappas.std(ddof=1),
        'n_pairs': len(kappas),
        'min':     kappas.min(),
        'max':     kappas.max(),
    }
    return summary, mat


def display_kappa(summary, mat, title):
    print(f'\n{"+"*72}\n  {title}\n{"+"*72}')
    print(f"Mean pairwise quadratic-weighted kappa: "
          f"{summary['mean']:.3f} \u00b1 {summary['sd']:.3f}  "
          f"(n={summary['n_pairs']} pairs, "
          f"range {summary['min']:.3f}\u2013{summary['max']:.3f})")
    display(mat.round(3))

---
## 1 · Label-base Human Analysis
9 raters labeling from text/label criteria.

In [ ]:
ALL_HUMAN_RAW_PATH = '/Users/paritt.w/Desktop/STRAIGHT/Result/All_human_raw_label.xlsx'
ALPHA_THRESH = 0.2
OBSERVERS    = ['Aimmy', 'Big', 'Boo', 'Graph', 'Mook', 'Jan', 'Jern', 'Pear', 'Ploy']

# Row 0 is a sub-header (X1/X2/X3 labels) — skip it
df_lb = pd.read_excel(ALL_HUMAN_RAW_PATH).iloc[1:].reset_index(drop=True).copy()

# Rename columns: GT/GT.1/GT.2 → GT_X1/GT_X2/GT_X3, ObsName/ObsName.1/ObsName.2 → Obs_X1/X2/X3
rename_map = {'GT': 'GT_X1', 'GT.1': 'GT_X2', 'GT.2': 'GT_X3'}
for obs in OBSERVERS:
    rename_map[obs]        = f'{obs}_X1'
    rename_map[f'{obs}.1'] = f'{obs}_X2'
    rename_map[f'{obs}.2'] = f'{obs}_X3'
df_lb = df_lb.rename(columns=rename_map)

# Convert all coordinate columns to numeric
all_coord_cols = ['GT_X1','GT_X2','GT_X3'] + \
                 [f'{o}_{c}' for o in OBSERVERS for c in ['X1','X2','X3']]
for col in all_coord_cols:
    df_lb[col] = pd.to_numeric(df_lb[col], errors='coerce')

# ── Alpha & label computation ─────────────────────────────────────────────────
def _alpha(x1, x2, x3):
    dl = x2 - x3;  dr = x3 - x1;  denom = dl + dr
    return float((dr - dl) / denom) if denom != 0 else 0.0

def _label_from_alpha(a, t=ALPHA_THRESH):
    if a >  t: return 'Left rotate'
    if a < -t: return 'Right rotate'
    return 'Normal'

df_lb['GT_Alpha'] = df_lb.apply(
    lambda r: _alpha(r['GT_X1'], r['GT_X2'], r['GT_X3']), axis=1)
df_lb['GT_Label'] = df_lb['GT_Alpha'].apply(_label_from_alpha)

for obs in OBSERVERS:
    df_lb[f'{obs}_Alpha'] = df_lb.apply(
        lambda r, o=obs: _alpha(r[f'{o}_X1'], r[f'{o}_X2'], r[f'{o}_X3']), axis=1)
    df_lb[f'{obs}_Label'] = df_lb[f'{obs}_Alpha'].apply(_label_from_alpha)

# ── Build df_label in the same format expected by ICC / metrics helpers ───────
df_label = df_lb[['Image']].copy()
df_label['GT'] = df_lb['GT_Label']
for i, obs in enumerate(OBSERVERS):
    df_label[f'R{i+1:02d}'] = df_lb[f'{obs}_Label']
label_raters = [f'R{i+1:02d}' for i in range(len(OBSERVERS))]

print(f'Label-base raw: {len(df_lb)} images, {len(OBSERVERS)} observers: {OBSERVERS}')
print('\nGT label distribution:')
print(df_lb['GT_Label'].value_counts())

In [ ]:
icc_label = compute_icc(df_label, label_raters)
display_icc(icc_label,
            'Label-base — ICC among human raters (Left=1, Normal=2, Right=3)')

In [ ]:
kappa_label_summary, kappa_label_mat = compute_weighted_kappa(df_label, label_raters)
display_kappa(kappa_label_summary, kappa_label_mat,
              'Label-base \u2014 weighted kappa among human raters (quadratic)')

In [ ]:
metrics_label = build_metrics_table(df_label, label_raters)
mean_l, std_l = display_with_summary(
    metrics_label,
    'Label-base — Per-Rater Performance vs Ground Truth (macro-averaged, 3-class)'
)

In [ ]:
# ── Per-observer coordinate MAE ± SD vs GT ───────────────────────────────────
mae_records = []
for obs in OBSERVERS:
    ex1 = (df_lb['GT_X1'] - df_lb[f'{obs}_X1']).abs()
    ex2 = (df_lb['GT_X2'] - df_lb[f'{obs}_X2']).abs()
    ex3 = (df_lb['GT_X3'] - df_lb[f'{obs}_X3']).abs()
    eavg = (ex1 + ex2 + ex3) / 3
    mae_records.append({
        'Observer':  obs,
        'MAE X1':    round(ex1.mean(),  2),  'SD X1':   round(ex1.std(ddof=1),  2),
        'MAE X2':    round(ex2.mean(),  2),  'SD X2':   round(ex2.std(ddof=1),  2),
        'MAE X3':    round(ex3.mean(),  2),  'SD X3':   round(ex3.std(ddof=1),  2),
        'MAE Avg':   round(eavg.mean(), 2),  'SD Avg':  round(eavg.std(ddof=1), 2),
    })

human_mae_df = pd.DataFrame(mae_records).set_index('Observer')

# Summary (mean ± SD across observers)
hmean = human_mae_df.mean()
hstd  = human_mae_df.std(ddof=1)
summary_row = {c: f'{hmean[c]:.2f} ± {hstd[c]:.2f}' for c in human_mae_df.columns}

print('\n' + '+'*72)
print('  Label-base Human — Coordinate Prediction MAE ± SD (pixels, N=100)')
print('+'*72)
display(pd.concat([
    human_mae_df.round(2).astype(str),
    pd.DataFrame([summary_row], index=['Mean ± SD'])
]))

---
## 2 · Visual-base Human Analysis
20 raters labeling by visual inspection.

In [ ]:
df_visual, visual_raters, visual_rater_names = load_and_clean(VISUAL_CSV)
print(f'Visual-base: {len(df_visual)} images, {len(visual_raters)} raters: {visual_raters}')
print('\nGT distribution:')
print(df_visual['GT'].value_counts())

In [ ]:
icc_visual = compute_icc(df_visual, visual_raters)
display_icc(icc_visual,
            'Visual-base — ICC among human raters (Left=1, Normal=2, Right=3)')

In [ ]:
kappa_visual_summary, kappa_visual_mat = compute_weighted_kappa(df_visual, visual_raters)
display_kappa(kappa_visual_summary, kappa_visual_mat,
              'Visual-base \u2014 weighted kappa among human raters (quadratic)')

In [ ]:
# ── R-code -> rater-name mapping ──────────────────────────────────────────────
# Label-base R-codes follow the OBSERVERS list; Visual-base come from the CSV
# column order (captured by load_and_clean as visual_rater_names).
label_rater_names = dict(zip(label_raters, OBSERVERS))

print('+'*72)
print('  R-code \u2192 rater name')
print('+'*72)

print(f'\nLabel-base ({len(label_raters)} raters):')
display(pd.DataFrame(
    {'Rater name': [label_rater_names[r] for r in label_raters]},
    index=label_raters).rename_axis('R-code'))

print(f'\nVisual-base ({len(visual_raters)} raters):')
display(pd.DataFrame(
    {'Rater name': [visual_rater_names[r] for r in visual_raters]},
    index=visual_raters).rename_axis('R-code'))

In [ ]:
metrics_visual = build_metrics_table(df_visual, visual_raters)
mean_v, std_v = display_with_summary(
    metrics_visual,
    'Visual-base — Per-Rater Performance vs Ground Truth (macro-averaged, 3-class)'
)

---
## 3 · Human Summary

In [ ]:
METRIC_COLS = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1']

def fmt_icc(icc_df, icc_type):
    row = icc_df[icc_df['Type'] == icc_type].iloc[0]
    lo, hi = row['CI95%']
    return f"{row['ICC']:.3f} (95% CI {lo:.3f}\u2013{hi:.3f})"

rows = {
    'Dataset':         ['Label-base', 'Visual-base'],
    'N raters':        [len(label_raters), len(visual_raters)],
    'N images':        [len(df_label), len(df_visual)],
    'ICC2 (single)':   [fmt_icc(icc_label,  'ICC2'),  fmt_icc(icc_visual, 'ICC2')],
    'ICC2k (average)': [fmt_icc(icc_label,  'ICC2k'), fmt_icc(icc_visual, 'ICC2k')],
}
for col in METRIC_COLS:
    rows[col] = [
        f'{mean_l[col]:.3f} \u00b1 {std_l[col]:.3f}',
        f'{mean_v[col]:.3f} \u00b1 {std_v[col]:.3f}',
    ]

human_summary = pd.DataFrame(rows).set_index('Dataset')
print('\n' + '+='*36)
print('  Human Summary  (metrics = Mean \u00b1 SD across all raters)')
print('+='*36)
display(human_summary)

---
## 4 · Model Inference Setup
Load STRAIGHT (regression) and Classification CNN models.

In [ ]:
import torch
from torch import nn
import timm
from timm import create_model
import cv2
import os

In [ ]:
class HRNetCustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hrnet = timm.create_model('hrnet_w48', pretrained=True, features_only=True)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(1024 * 16 * 16, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 64)
        self.fc6 = nn.Linear(64, 3)

    def forward(self, x):
        features = self.hrnet(x)
        x = features[-1]
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        return self.fc6(x)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

rotation_model = HRNetCustomModel().to(device)
rotation_model.load_state_dict(torch.load(
    '/Users/paritt.w/Desktop/STRAIGHT/Model/best_HR+reg3(0.870).pth',
    weights_only=True, map_location=device
))
rotation_model.eval()

classification_model = create_model('hrnet_w48', pretrained=True, num_classes=3)
classification_model.load_state_dict(torch.load(
    '/Users/paritt.w/Desktop/STRAIGHT/Model/Best_classification3.pth',
    map_location=device
))
classification_model.to(device)
classification_model.eval()

print('Both models loaded.')

In [ ]:
IMAGE_DIR    = '/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/'
TEST_CSV     = '/Users/paritt.w/Desktop/STRAIGHT/Rotation images/Test/Test.csv'

# Map model output labels -> human CSV format (Rt/Lt are swapped relative to human labels)
MODEL_TO_HUMAN = {
    'Lt Rotate':   'Right rotate',
    'No Rotation': 'Normal',
    'Rt Rotate':   'Left rotate',
}


def _load_img(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_NEAREST)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float64) / 255.0


def _to_tensor(img_01):
    arr = np.transpose(img_01, (2, 0, 1))[np.newaxis]
    return torch.from_numpy(arr).float().to(device)


def _alpha_to_class(x1, x2, x3, threshold=0.2):
    dist_l = round(x2 - x3)
    dist_r = round(x3 - x1)
    alpha  = (dist_r - dist_l) / (dist_r + dist_l)
    if alpha < -threshold: return 'Lt Rotate'
    if alpha >  threshold: return 'Rt Rotate'
    return 'No Rotation'


def _compute_alpha(x1, x2, x3):
    dist_l = x2 - x3
    dist_r = x3 - x1
    return (dist_r - dist_l) / (dist_r + dist_l)


def run_straight(image_path):
    """Returns (class_label, pred_x1, pred_x2, pred_x3, alpha)."""
    img = _load_img(image_path)
    inp = _to_tensor(img)
    with torch.no_grad():
        pred = rotation_model(inp)[0].cpu().numpy()
    cls   = _alpha_to_class(pred[0], pred[1], pred[2])
    alpha = _compute_alpha(pred[0], pred[1], pred[2])
    return MODEL_TO_HUMAN[cls], float(pred[0]), float(pred[1]), float(pred[2]), round(float(alpha), 4)


def run_classification(image_path):
    """Returns (class_label, rotation_score) where score = max(P_left, P_right)."""
    img = _load_img(image_path)
    inp = _to_tensor(img)
    with torch.no_grad():
        out   = classification_model(inp)
        probs = torch.softmax(out, dim=1)[0].cpu().numpy()
        idx   = torch.argmax(out, dim=1).item()
    # class order from training: ["Rt Rotate", "No Rotation", "Lt Rotate"]
    cls   = ['Rt Rotate', 'No Rotation', 'Lt Rotate'][idx]
    score = float(max(probs[0], probs[2]))   # max(P_Rt, P_Lt)
    return MODEL_TO_HUMAN[cls], round(score, 4)

---
## 5 · Run Inference & Save model_result.csv
Same 100 images as the human CSVs (GT from Label-base CSV).

In [ ]:
import time

df_test = pd.read_csv(TEST_CSV).set_index('Image')

result_rows    = []
times_straight = []
times_classify = []

for _, row in df_label.iterrows():
    img_path = IMAGE_DIR + row['Image']
    gt = df_test.loc[row['Image']]

    t0 = time.perf_counter()
    st_label, st_x1, st_x2, st_x3, st_alpha = run_straight(img_path)
    times_straight.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    cl_pred, cl_score = run_classification(img_path)
    times_classify.append(time.perf_counter() - t0)

    result_rows.append({
        'Image':          row['Image'],
        'GT':             row['GT'],
        'GT_X1':          gt['X1'],
        'GT_X2':          gt['X2'],
        'GT_X3':          gt['X3'],
        'STRAIGHT':       st_label,
        'ST_X1':          round(st_x1,    2),
        'ST_X2':          round(st_x2,    2),
        'ST_X3':          round(st_x3,    2),
        'ST_Alpha':       st_alpha,
        'Classification': cl_pred,
        'CL_Score':       cl_score,
    })

df_model = pd.DataFrame(result_rows)

save_path = f'{RESULT_DIR}/model_result.csv'
df_model.to_csv(save_path, index=False)
print(f'Saved {len(df_model)} rows → {save_path}')
display(df_model.head(10))

In [ ]:
def timing_stats(t_list):
    a = np.array(t_list)
    return {
        'Mean (s)': a.mean(),
        'SD (s)':   a.std(ddof=1),
        'Min (s)':  a.min(),
        'Max (s)':  a.max(),
        'Mean (ms)': a.mean() * 1000,
        'SD (ms)':   a.std(ddof=1) * 1000,
        'Min (ms)':  a.min() * 1000,
        'Max (ms)':  a.max() * 1000,
    }

timing_df = pd.DataFrame([
    {'Model': 'STRAIGHT',       **timing_stats(times_straight)},
    {'Model': 'Classification', **timing_stats(times_classify)},
]).set_index('Model')

# Display in milliseconds (more readable)
display_cols = ['Mean (ms)', 'SD (ms)', 'Min (ms)', 'Max (ms)']
print('\n' + '+'*60)
print('  Inference Time per Image (milliseconds, N=100)')
print('+'*60)
display(timing_df[display_cols].round(2))

In [ ]:
def col_mae_sd(gt_col, pred_col):
    errs = np.abs(df_model[gt_col].values - df_model[pred_col].values)
    return float(errs.mean()), float(errs.std(ddof=1))

mae_x1, sd_x1 = col_mae_sd('GT_X1', 'ST_X1')
mae_x2, sd_x2 = col_mae_sd('GT_X2', 'ST_X2')
mae_x3, sd_x3 = col_mae_sd('GT_X3', 'ST_X3')
mae_avg = np.mean([mae_x1, mae_x2, mae_x3])
sd_avg  = np.mean([sd_x1,  sd_x2,  sd_x3])

mae_df = pd.DataFrame([{
    'MAE X1 (px)':  round(mae_x1,  2),
    'SD X1 (px)':   round(sd_x1,   2),
    'MAE X2 (px)':  round(mae_x2,  2),
    'SD X2 (px)':   round(sd_x2,   2),
    'MAE X3 (px)':  round(mae_x3,  2),
    'SD X3 (px)':   round(sd_x3,   2),
    'MAE Avg (px)': round(mae_avg, 2),
    'SD Avg (px)':  round(sd_avg,  2),
}], index=['STRAIGHT'])

print('\n' + '+'*65)
print('  STRAIGHT — Coordinate Prediction MAE ± SD (pixels, N=100)')
print('+'*65)
display(mae_df)

In [ ]:
df_model = pd.read_csv(f'{RESULT_DIR}/model_result.csv')
print(f'Model results: {len(df_model)} images')
print('\nGT distribution:')
print(df_model['GT'].value_counts())

---
## 6 · Model Metrics

In [ ]:
y_true = df_model['GT'].tolist()

m_straight = rater_metrics(y_true, df_model['STRAIGHT'].tolist())
m_classify  = rater_metrics(y_true, df_model['Classification'].tolist())

model_metrics = pd.DataFrame([
    {'Model': 'STRAIGHT',       **m_straight},
    {'Model': 'Classification', **m_classify},
]).set_index('Model')[['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1']]

print('\n' + '+'*72)
print('  Model Performance vs Ground Truth (macro-averaged, 3-class)')
print('+'*72)
display(model_metrics.round(3))

---
## 6b · Confusion Matrices

In [ ]:
import seaborn as sns

cm_straight = confusion_matrix(y_true, df_model['STRAIGHT'].tolist(),       labels=CLASSES)
cm_classify = confusion_matrix(y_true, df_model['Classification'].tolist(), labels=CLASSES)

tick_labels = ['Left\nrotate', 'Normal', 'Right\nrotate']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

sns.heatmap(cm_straight, annot=True, fmt='d', cmap='Blues', vmin=0, vmax=34,
            xticklabels=tick_labels, yticklabels=tick_labels,
            ax=axes[0], annot_kws={'size': 13})
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, va='center')
axes[0].set_title('STRAIGHT', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Predicted',    fontsize=11)
axes[0].set_ylabel('Ground Truth', fontsize=11)

sns.heatmap(cm_classify, annot=True, fmt='d', cmap='Greens', vmin=0, vmax=34,
            xticklabels=tick_labels, yticklabels=tick_labels,
            ax=axes[1], annot_kws={'size': 13})
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0, va='center')
axes[1].set_title('CNN Classification', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Predicted',         fontsize=11)
axes[1].set_ylabel('Ground Truth',      fontsize=11)

plt.suptitle('Confusion Matrix',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../Fig', exist_ok=True)
plt.savefig('../Fig/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved → ../Fig/confusion_matrices.png')

---
## 7 · Full Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {'Method': 'Human Label-base',
     **{c: f'{mean_l[c]:.3f} \u00b1 {std_l[c]:.3f}' for c in METRIC_COLS}},
    {'Method': 'Human Visual-base',
     **{c: f'{mean_v[c]:.3f} \u00b1 {std_v[c]:.3f}' for c in METRIC_COLS}},
    {'Method': 'STRAIGHT',
     **{c: f'{m_straight[c]:.3f}' for c in METRIC_COLS}},
    {'Method': 'Classification CNN',
     **{c: f'{m_classify[c]:.3f}'  for c in METRIC_COLS}},
]).set_index('Method')

print('\n' + '+='*40)
print('  Full Comparison: Human vs Models')
print('+='*40)
display(comparison)

---
## 7b · Per-Class Metrics Breakdown
Sensitivity, Specificity, Precision, F1, Accuracy for each class (Left rotate / Normal / Right rotate) + macro average.  
Human results reported as Mean ± SD across raters.

In [ ]:
_PC_CLASSES = ['Left rotate', 'Normal', 'Right rotate', 'Average']
_PC_METRICS = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1']


def _per_class_dict(y_true, y_pred):
    cm_mat = confusion_matrix(y_true, y_pred, labels=CLASSES)
    total  = cm_mat.sum()
    out = {}
    for i, cls in enumerate(CLASSES):
        TP = cm_mat[i, i]
        FP = cm_mat[:, i].sum() - TP
        FN = cm_mat[i, :].sum() - TP
        TN = total - TP - FP - FN
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        f1   = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
        out[cls] = {
            'Accuracy':    (TP + TN) / total,
            'Sensitivity': sens,
            'Specificity': spec,
            'Precision':   prec,
            'F1':          f1,
        }
    out['Average'] = {
        'Accuracy':    accuracy_score(y_true, y_pred),
        'Sensitivity': np.mean([out[c]['Sensitivity'] for c in CLASSES]),
        'Specificity': np.mean([out[c]['Specificity'] for c in CLASSES]),
        'Precision':   np.mean([out[c]['Precision']   for c in CLASSES]),
        'F1':          np.mean([out[c]['F1']           for c in CLASSES]),
    }
    return out


def model_pc_table(y_true, y_pred):
    d = _per_class_dict(y_true, y_pred)
    return pd.DataFrame(
        [{'Class': cls, **{m: round(d[cls][m], 3) for m in _PC_METRICS}}
         for cls in _PC_CLASSES]
    ).set_index('Class')


def human_pc_table(df, rater_cols):
    store = {cls: {m: [] for m in _PC_METRICS} for cls in _PC_CLASSES}
    for rater in rater_cols:
        mask = df[rater].notna() & df['GT'].notna()
        d = _per_class_dict(df.loc[mask, 'GT'].tolist(), df.loc[mask, rater].tolist())
        for cls in _PC_CLASSES:
            for m in _PC_METRICS:
                store[cls][m].append(d[cls][m])
    rows = []
    for cls in _PC_CLASSES:
        row = {'Class': cls}
        for m in _PC_METRICS:
            vals = np.array(store[cls][m])
            row[m] = f'{vals.mean():.3f} ± {vals.std(ddof=1):.3f}'
        rows.append(row)
    return pd.DataFrame(rows).set_index('Class')


# ── Compute ───────────────────────────────────────────────────────────────────
pc_straight = model_pc_table(y_true, df_model['STRAIGHT'].tolist())
pc_classify = model_pc_table(y_true, df_model['Classification'].tolist())
pc_label    = human_pc_table(df_label,  label_raters)
pc_visual   = human_pc_table(df_visual, visual_raters)

sep = '─' * 72
for title, tbl in [
    ('STRAIGHT — Per-Class Metrics',                       pc_straight),
    ('Classification CNN — Per-Class Metrics',             pc_classify),
    ('Human Label-base — Per-Class Metrics  (Mean ± SD)',  pc_label),
    ('Human Visual-base — Per-Class Metrics  (Mean ± SD)', pc_visual),
]:
    print(f'\n{sep}\n  {title}\n{sep}')
    display(tbl)

---
## 7c · Statistical Tests
**Cochran's Q** (omnibus) → **Pairwise McNemar's** (Bonferroni-corrected) → **Accuracy difference 95% CI** (bootstrap).  
Human methods use per-image majority-vote label.

In [ ]:
from scipy.stats import chi2 as chi2_dist, binom
from itertools import combinations

# ── Per-image correct/incorrect ───────────────────────────────────────────────
# Authoritative GT from Test.csv (same order as df_model / df_label)
gt_test = df_model['GT'].values

correct_straight = (gt_test == df_model['STRAIGHT'].values).astype(int)
correct_classify = (gt_test == df_model['Classification'].values).astype(int)

def majority_vote_labels(df, rater_cols):
    votes = []
    for _, row in df.iterrows():
        preds = [row[r] for r in rater_cols if pd.notna(row[r])]
        votes.append(pd.Series(preds).mode()[0])
    return np.array(votes)

mv_label  = majority_vote_labels(df_label,  label_raters)
mv_visual = majority_vote_labels(df_visual, visual_raters)

correct_label  = (gt_test == mv_label).astype(int)
correct_visual = (gt_test == mv_visual).astype(int)

method_names   = ['STRAIGHT', 'Classification CNN', 'Human Label-base', 'Human Visual-base']
correct_matrix = np.column_stack([correct_straight, correct_classify,
                                   correct_label,    correct_visual])

print('Accuracy vs Test.csv GT (majority vote for humans):')
for name, col in zip(method_names, correct_matrix.T):
    print(f'  {name:<26}: {col.mean():.3f}  ({int(col.sum())}/{len(col)})')

# ── Cochran's Q (omnibus) ─────────────────────────────────────────────────────
def cochrans_q_test(data):
    n, k = data.shape
    L = data.sum(axis=0)
    R = data.sum(axis=1)
    denom = k * R.sum() - (R**2).sum()
    if denom == 0:
        return 0.0, k - 1, 1.0
    Q = k * (k - 1) * ((L - L.mean()) ** 2).sum() / denom
    return float(Q), k - 1, float(1 - chi2_dist.cdf(Q, df=k - 1))

Q_stat, Q_df, Q_p = cochrans_q_test(correct_matrix)
verdict = 'SIGNIFICANT — proceed to pairwise tests' if Q_p < 0.05 \
          else 'NOT significant — no evidence methods differ'
print(f"\nCochran's Q = {Q_stat:.3f},  df = {Q_df},  p = {Q_p:.4f}  →  {verdict}")

# ── Pairwise McNemar's (Bonferroni) ───────────────────────────────────────────
def mcnemar_test(a, b):
    b_cnt  = int(((a == 1) & (b == 0)).sum())
    c_cnt  = int(((a == 0) & (b == 1)).sum())
    n_disc = b_cnt + c_cnt
    if n_disc == 0:
        return 0.0, 1.0, b_cnt, c_cnt
    if n_disc <= 25:                        # exact binomial
        p = float(min(1.0, 2 * binom.cdf(min(b_cnt, c_cnt), n_disc, 0.5)))
        return float(min(b_cnt, c_cnt)), p, b_cnt, c_cnt
    stat = (abs(b_cnt - c_cnt) - 1) ** 2 / n_disc   # with continuity correction
    return float(stat), float(1 - chi2_dist.cdf(stat, df=1)), b_cnt, c_cnt

pairs         = list(combinations(range(4), 2))
n_comparisons = len(pairs)

mc_rows = []
for i, j in pairs:
    stat, p, b, c = mcnemar_test(correct_matrix[:, i], correct_matrix[:, j])
    p_bonf = min(p * n_comparisons, 1.0)
    mc_rows.append({
        'Method A':       method_names[i],
        'Method B':       method_names[j],
        'b (A✓ B✗)':     b,
        'c (A✗ B✓)':     c,
        'Statistic':      round(stat,   3),
        'p-value':        round(p,      4),
        'p (Bonferroni)': round(p_bonf, 4),
        'Sig.*':          '✓' if p_bonf < 0.05 else '—',
    })

mcnemar_df = pd.DataFrame(mc_rows)
print(f"\nPairwise McNemar's  (Bonferroni n={n_comparisons};  * = p < 0.05 after correction)")
display(mcnemar_df.set_index(['Method A', 'Method B']))

# ── Accuracy difference with 95% bootstrap CI ────────────────────────────────
def acc_diff_ci(a, b, n_boot=10000, seed=42):
    np.random.seed(seed)
    n     = len(a)
    diffs = [a[idx].mean() - b[idx].mean()
             for idx in (np.random.choice(n, n, replace=True) for _ in range(n_boot))]
    obs   = float(a.mean() - b.mean())
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return obs, float(lo), float(hi)

ci_rows = []
for i, j in pairs:
    diff, lo, hi = acc_diff_ci(correct_matrix[:, i], correct_matrix[:, j])
    ci_rows.append({
        'Method A':   method_names[i],
        'Method B':   method_names[j],
        'Acc A':      round(correct_matrix[:, i].mean(), 3),
        'Acc B':      round(correct_matrix[:, j].mean(), 3),
        'Diff (A−B)': round(diff, 3),
        '95% CI':     f'[{lo:.3f}, {hi:.3f}]',
        'Conclusion': 'No sig. difference' if lo <= 0 <= hi else 'Significant difference',
    })

acc_ci_df = pd.DataFrame(ci_rows)
print('\nAccuracy difference  95% bootstrap CI  (A − B,  n_boot=10,000)')
display(acc_ci_df.set_index(['Method A', 'Method B']))

---
## 8 · Radar Plot

In [ ]:
metrics_names = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1']

label_vals    = [mean_l[m]       for m in metrics_names]
visual_vals   = [mean_v[m]       for m in metrics_names]
straight_vals = [m_straight[m]   for m in metrics_names]
class_vals    = [m_classify[m]   for m in metrics_names]

num_vars = len(metrics_names)
angles   = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

# Close each polygon
def close(vals): return vals + vals[:1]
label_vals, visual_vals = close(label_vals), close(visual_vals)
straight_vals, class_vals = close(straight_vals), close(class_vals)
angles = angles + angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'), dpi=150)

ax.plot(angles, straight_vals,  'o-',  lw=2, label='STRAIGHT',            color='#1100FF')
ax.fill(angles, straight_vals,  alpha=0.12,                                color='#1100FF')

ax.plot(angles, class_vals,     'o-',  lw=2, label='Classification CNN',   color='#4BC86D')
ax.fill(angles, class_vals,     alpha=0.12,                                color='#4BC86D')

ax.plot(angles, label_vals,     's--', lw=2, label='Human Label-base',     color='#FF8C00')
ax.fill(angles, label_vals,     alpha=0.10,                                color='#FF8C00')

ax.plot(angles, visual_vals,    's--', lw=2, label='Human Visual-base',    color='#FF0000')
ax.fill(angles, visual_vals,    alpha=0.10,                                color='#FF0000')

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_names, size=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=9)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=11)
ax.grid(True, linestyle='--', alpha=0.7)
ax.set_title('Human vs Model Performance', size=14, fontweight='bold', pad=20)

plt.tight_layout()

fig_dir = '../Fig'
os.makedirs(fig_dir, exist_ok=True)
plt.savefig(f'{fig_dir}/radar_human_vs_model.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved → ../Fig/radar_human_vs_model.png')

---
## 8b · ROC Curves with 95% CI (Bootstrap)

In [ ]:
from sklearn.metrics import roc_curve, auc

# ── Binary GT ────────────────────────────────────────────
# Model GT: rotation = 1, Normal = 0
y_true_model = (df_model['GT'] != 'Normal').astype(int).values
# Human GT derived from raw coordinates (GT_Label in df_lb)
y_true_human = (df_lb['GT_Label'] != 'Normal').astype(int).values

# ── Bootstrap AUC 95% CI ──────────────────────────────────────
def bootstrap_auc(y_true, y_scores, n_bootstraps=10000, seed=42):
    np.random.seed(seed)
    y_true, y_scores = np.array(y_true), np.array(y_scores)
    fpr0, tpr0, _ = roc_curve(y_true, y_scores)
    auc_orig = auc(fpr0, tpr0)
    aucs = []
    for _ in range(n_bootstraps):
        idx = np.random.choice(len(y_true), size=len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        f, t, _ = roc_curve(y_true[idx], y_scores[idx])
        aucs.append(auc(f, t))
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    return auc_orig, lo, hi

# ── Bootstrap AUC 95% CI over raters (same percentile-bootstrap method, applied to the 9 human raters) ──
def bootstrap_auc_over_raters(auc_per_obs, tpr_per_obs, n_bootstraps=10000, seed=42):
    np.random.seed(seed)
    auc_per_obs = np.array(auc_per_obs)
    tpr_per_obs = np.array(tpr_per_obs)
    n_raters = len(auc_per_obs)
    boot_aucs = np.empty(n_bootstraps)
    boot_curves = np.empty((n_bootstraps, tpr_per_obs.shape[1]))
    for i in range(n_bootstraps):
        idx = np.random.choice(n_raters, size=n_raters, replace=True)
        boot_aucs[i] = auc_per_obs[idx].mean()
        boot_curves[i] = tpr_per_obs[idx].mean(axis=0)
    auc_lo, auc_hi = np.percentile(boot_aucs, [2.5, 97.5])
    tpr_lo = np.percentile(boot_curves, 2.5, axis=0)
    tpr_hi = np.percentile(boot_curves, 97.5, axis=0)
    return float(auc_per_obs.mean()), auc_lo, auc_hi, tpr_lo, tpr_hi

# ── Model scores ────────────────────────────────────────────
y_scores_straight = df_model['ST_Alpha'].abs().values
y_scores_classify = df_model['CL_Score'].values

fpr_st, tpr_st, _ = roc_curve(y_true_model, y_scores_straight)
fpr_cl, tpr_cl, _ = roc_curve(y_true_model, y_scores_classify)

print('Bootstrapping STRAIGHT AUC...')
auc_st, ci_st_lo, ci_st_hi = bootstrap_auc(y_true_model, y_scores_straight)
print('Bootstrapping Classification AUC...')
auc_cl, ci_cl_lo, ci_cl_hi = bootstrap_auc(y_true_model, y_scores_classify)

# ── Human ROC: abs(alpha) per observer → mean curve + bootstrap 95% CI (same method as STRAIGHT/CNN) ──
fpr_grid    = np.linspace(0, 1, 300)
tpr_per_obs = []
auc_per_obs = []

for obs in OBSERVERS:
    y_s = df_lb[f'{obs}_Alpha'].abs().values
    f, t, _ = roc_curve(y_true_human, y_s)
    tpr_per_obs.append(np.interp(fpr_grid, f, t))
    auc_per_obs.append(auc(f, t))

print('Bootstrapping Human AUC (resampled over the 9 raters)...')
auc_mean, auc_ci_lo, auc_ci_hi, tpr_ci_lo, tpr_ci_hi = bootstrap_auc_over_raters(auc_per_obs, tpr_per_obs)
tpr_mean = np.mean(tpr_per_obs, axis=0)
auc_min  = float(np.min(auc_per_obs))
auc_max  = float(np.max(auc_per_obs))

print(f'\nAUC — STRAIGHT:       {auc_st:.3f} (95% CI {ci_st_lo:.3f}–{ci_st_hi:.3f})')
print(f'AUC — Classification: {auc_cl:.3f} (95% CI {ci_cl_lo:.3f}–{ci_cl_hi:.3f})')
print(f'AUC — Human mean:     {auc_mean:.3f} (95% CI {auc_ci_lo:.3f}–{auc_ci_hi:.3f})')
print(f'      individual rater range: {auc_min:.3f}–{auc_max:.3f} (observed spread, not the reported CI)')

# ── Plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7), dpi=150)

# Human: gray fill (bootstrap 95% CI band) + gray dashed (mean)
ax.fill_between(fpr_grid, tpr_ci_lo, tpr_ci_hi,
                color='gray', alpha=0.25, label='Human 95% CI (bootstrap over raters)')
ax.plot(fpr_grid, tpr_mean, color='gray', linestyle='--', linewidth=2,
        label=f'Human (mean)        AUC {auc_mean:.3f} (95% CI {auc_ci_lo:.3f}–{auc_ci_hi:.3f})')

# STRAIGHT
ax.plot(fpr_st, tpr_st, color='black', linestyle='-', linewidth=2,
        label=f'STRAIGHT               AUC {auc_st:.3f} (95% CI {ci_st_lo:.3f}–{ci_st_hi:.3f})')

# Classification CNN
ax.plot(fpr_cl, tpr_cl, color='black', linestyle='--', linewidth=2,
        label=f'Classification CNN  AUC {auc_cl:.3f} (95% CI {ci_cl_lo:.3f}–{ci_cl_hi:.3f})')

ax.set_xlabel('1 – Specificity', fontsize=12)
ax.set_ylabel('Sensitivity',     fontsize=12)
ax.set_xlim([0, 1]);  ax.set_ylim([0, 1.05])
ax.legend(loc='lower right', fontsize=9.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
os.makedirs('../Fig', exist_ok=True)
plt.savefig('../Fig/roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved → ../Fig/roc_curves.png')

---
## 9 · Save All Results to CSV

In [ ]:
RESULT_DIR = '../Result/Analyze_Results'
os.makedirs(RESULT_DIR, exist_ok=True)

def save_icc_csv(icc_df, path):
    df = icc_df.copy()
    df['CI95%_low']  = df['CI95%'].apply(lambda x: round(x[0], 3))
    df['CI95%_high'] = df['CI95%'].apply(lambda x: round(x[1], 3))
    df.drop(columns=['CI95%']).round(3).to_csv(path, index=False)

def numeric_summary(mean_s, std_s):
    row = {}
    for c in mean_s.index:
        row[f'{c}_mean'] = round(mean_s[c], 3)
        row[f'{c}_sd']   = round(std_s[c],  3)
    return row

# 1. Label-base per-rater metrics
metrics_label.round(3).to_csv(f'{RESULT_DIR}/label_base_rater_metrics.csv')

# 2. Visual-base per-rater metrics
metrics_visual.round(3).to_csv(f'{RESULT_DIR}/visual_base_rater_metrics.csv')

# 3–4. ICC tables
save_icc_csv(icc_label,  f'{RESULT_DIR}/icc_label_base.csv')
save_icc_csv(icc_visual, f'{RESULT_DIR}/icc_visual_base.csv')

# 5. Human summary (numeric)
human_numeric = pd.DataFrame([
    {'Dataset': 'Label-base',  **numeric_summary(mean_l, std_l)},
    {'Dataset': 'Visual-base', **numeric_summary(mean_v, std_v)},
]).set_index('Dataset')
human_numeric.to_csv(f'{RESULT_DIR}/human_summary_numeric.csv')

# 6. Model predictions + coordinates + scores
df_model.to_csv(f'{RESULT_DIR}/model_result.csv', index=False)

# 7. STRAIGHT coordinate predictions with per-image error
coord_df = df_model[['Image', 'GT_X1', 'GT_X2', 'GT_X3',
                      'ST_X1', 'ST_X2', 'ST_X3']].copy()
coord_df['Err_X1'] = (coord_df['GT_X1'] - coord_df['ST_X1']).abs().round(2)
coord_df['Err_X2'] = (coord_df['GT_X2'] - coord_df['ST_X2']).abs().round(2)
coord_df['Err_X3'] = (coord_df['GT_X3'] - coord_df['ST_X3']).abs().round(2)
coord_df.to_csv(f'{RESULT_DIR}/straight_coordinates.csv', index=False)

# 8. STRAIGHT MAE summary
mae_df.to_csv(f'{RESULT_DIR}/straight_mae.csv')

# 8b. Human Label-base coordinate MAE ± SD per observer
human_mae_df.to_csv(f'{RESULT_DIR}/human_label_base_mae.csv')

# 9. Model classification metrics
model_metrics.round(3).to_csv(f'{RESULT_DIR}/model_metrics.csv')

# 10. Inference timing
timing_df.round(3).to_csv(f'{RESULT_DIR}/inference_timing.csv')

# 11. Full comparison
full_numeric = pd.DataFrame([
    {'Method': 'Human Label-base',   **{c: round(mean_l[c],     3) for c in METRIC_COLS}},
    {'Method': 'Human Visual-base',  **{c: round(mean_v[c],     3) for c in METRIC_COLS}},
    {'Method': 'STRAIGHT',           **{c: round(m_straight[c], 3) for c in METRIC_COLS}},
    {'Method': 'Classification CNN', **{c: round(m_classify[c], 3) for c in METRIC_COLS}},
]).set_index('Method')
full_numeric.to_csv(f'{RESULT_DIR}/full_comparison.csv')

# 12. ROC AUC summary
roc_auc_df = pd.DataFrame([
    {'Method': 'STRAIGHT',
     'AUC': round(auc_st, 3), 'CI95%_low': round(ci_st_lo, 3), 'CI95%_high': round(ci_st_hi, 3)},
    {'Method': 'Classification CNN',
     'AUC': round(auc_cl, 3), 'CI95%_low': round(ci_cl_lo, 3), 'CI95%_high': round(ci_cl_hi, 3)},
    {'Method': 'Human (mean)',
     'AUC': round(auc_mean, 3), 'CI95%_low': round(auc_ci_lo, 3), 'CI95%_high': round(auc_ci_hi, 3)},
]).set_index('Method')
roc_auc_df.to_csv(f'{RESULT_DIR}/roc_auc_summary.csv')

# 13–16. Per-class metrics breakdown
pc_straight.to_csv(f'{RESULT_DIR}/per_class_straight.csv')
pc_classify.to_csv(f'{RESULT_DIR}/per_class_classification.csv')
pc_label.to_csv(f'{RESULT_DIR}/per_class_human_label.csv')
pc_visual.to_csv(f'{RESULT_DIR}/per_class_human_visual.csv')

# 17. Cochran's Q summary
cochran_q_summary = pd.DataFrame([{
    'Q_statistic': round(Q_stat, 3),
    'df':          Q_df,
    'p_value':     round(Q_p, 4),
    'significant': Q_p < 0.05,
}])
cochran_q_summary.to_csv(f'{RESULT_DIR}/cochrans_q.csv', index=False)

# 18. Pairwise McNemar's
mcnemar_df.to_csv(f'{RESULT_DIR}/mcnemar_pairwise.csv', index=False)

# 19. Accuracy difference 95% CI
acc_ci_df.to_csv(f'{RESULT_DIR}/accuracy_diff_ci.csv', index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
saved = sorted(f for f in os.listdir(RESULT_DIR) if f.endswith('.csv'))
print(f'Saved {len(saved)} CSV files to {RESULT_DIR}/')
for f in saved:
    print(f'  {f}')